# 🔑 Solutions: Lesson 00 · Welcome, Setup & Your First Neural Network

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 0.1: Standardise the columns  &nbsp; `⭐`

Write `standardize(X)` that subtracts each column's mean and divides by its standard deviation
(so every column has mean 0 and std 1). **No loops.**  We do this to almost every dataset before training.


In [2]:
def standardize(X):
    return (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

In [3]:
# ▶ self-check
X = np.random.default_rng(0).normal(50, 10, size=(100, 3)) * [1, 5, 0.1]
Z = standardize(X)
dl.check("means ≈ 0", np.allclose(Z.mean(0), 0, atol=1e-6))
dl.check("stds ≈ 1", np.allclose(Z.std(0), 1, atol=1e-3))
dl.check("shape preserved", Z.shape == X.shape)

✅ means ≈ 0
✅ stds ≈ 1
✅ shape preserved


**Why this works.** `axis=0` collapses the rows, leaving one statistic per column; the `(features,)` result is broadcast across all rows.

---
### 🧩 Problem 0.2: A whole layer in one line  &nbsp; `⭐⭐`

A layer with `n_out` neurons computes `Z = X @ W + b` where `X` is `(batch, n_in)`, `W` is `(n_in, n_out)`,
and `b` is `(n_out,)`. Write `dense(X, W, b)` returning `Z`, **and** the function `relu(Z)` that
replaces negative numbers with 0 (use `np.maximum`).
Then compute the output of `relu(dense(X, W, b))` for the provided data.


In [4]:
def dense(X, W, b):
    return X @ W + b

def relu(Z):
    return np.maximum(0, Z)

rng = np.random.default_rng(1)
X = rng.normal(size=(5, 4)); W = rng.normal(size=(4, 3)); b = rng.normal(size=3)
out = relu(dense(X, W, b))

In [5]:
# ▶ self-check
dl.check("output shape is (5, 3)", out is not None and out.shape == (5, 3))
dl.check("no negative values", out is not None and (out >= 0).all())
dl.check("matches manual first neuron", out is not None and np.isclose(out[0, 0], max(0, X[0] @ W[:, 0] + b[0])))

✅ output shape is (5, 3)
✅ no negative values
✅ matches manual first neuron


**Why this works.** Matrix multiplication applies *every neuron to every example at once*. This single line is the engine of all neural networks.

---
### 🧩 Problem 0.3: One-hot encoding  &nbsp; `⭐⭐`

Classifiers often need labels as **one-hot** vectors: class `3` out of 5 classes becomes `[0,0,0,1,0]`.
Write `one_hot(labels, num_classes)` that returns an array of shape `(len(labels), num_classes)`: **without a Python loop over the labels**.


In [6]:
def one_hot(labels, num_classes):
    out = np.zeros((len(labels), num_classes))
    out[np.arange(len(labels)), labels] = 1
    return out

In [7]:
# ▶ self-check
oh = one_hot(np.array([3, 0, 4]), 5)
dl.check("shape (3,5)", oh is not None and oh.shape == (3, 5))
dl.check("row 0 is class 3", oh is not None and oh[0].argmax() == 3 and oh[0].sum() == 1)
dl.check("equals identity rows", oh is not None and np.array_equal(oh, np.eye(5)[[3, 0, 4]]))

✅ shape (3,5)
✅ row 0 is class 3
✅ equals identity rows


---
### 🧩 Problem 0.4: Predict the broadcast (shape reasoning)  &nbsp; `⭐`

Without running anything, decide the resulting shape of each expression, then fill the dictionary `answers`
with tuples. (Then run the check, and *also* run the expressions to confirm!)

* `a`: `np.ones((8, 1)) + np.ones((1, 5))`
* `b`: `np.ones((8, 5)) * np.ones(5)`
* `c`: `np.ones((8, 5)).sum(axis=1)`
* `d`: `np.ones((8, 5)).sum(axis=1, keepdims=True)`


In [8]:
answers = {"a": (8, 5), "b": (8, 5), "c": (8,), "d": (8, 1)}

In [9]:
# ▶ self-check
truth = {"a": (np.ones((8,1)) + np.ones((1,5))).shape, "b": (np.ones((8,5)) * np.ones(5)).shape,
         "c": np.ones((8,5)).sum(axis=1).shape, "d": np.ones((8,5)).sum(axis=1, keepdims=True).shape}
for k in truth:
    dl.check(f"shape of {k}", answers.get(k) == truth[k])

✅ shape of a
✅ shape of b
✅ shape of c
✅ shape of d
